# Altair basics

**Altair -- charts as a grammar, and linked selections.**

Altair makes you name the ENCODING: which column maps to x, to colour, to size, and what kind of data each one is (quantitative, ordinal, nominal, temporal). That is more typing than plotly for a quick chart, and it pays off the moment you want two charts to talk to each other.

**What it shows:**

- the encoding grammar, with the :Q / :N / :T type suffixes
- why declaring the type matters -- get it wrong and the axis is wrong
- a linked selection: brush one chart, filter the other

---

*Chapter:* `interactive` — plotly and altair, and when interactivity is worth it  
*Run the cells in order.* Every figure is also written to `viz/output/interactive/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save_html()` writes each chart into `viz/output/` as a standalone page you can open, email or embed. The chart also renders below the cell, because the cell ends by naming it.


In [ ]:
# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import altair as alt

from vizkit import save_html, sales, students

# Where save_html() files this lesson's output: viz/output/interactive/
LESSON = "interactive/altair_basics"


## The data

Altair works on tidy DataFrames — one row per observation, one column per variable — and does the grouping for you.


In [ ]:
data = students()


## 1. The grammar

The whole idea: you name the **encoding** — which column goes to x, to y, to colour, to tooltip — and Altair works out the axes, scales and legend. More typing than Plotly for one chart, and it pays off in the next section but one.


In [ ]:
# x, y, colour are ENCODINGS; :Q means quantitative, :N nominal (unordered).
chart = (
    alt.Chart(data)
    .mark_circle(size=60, opacity=0.6)
    .encode(
        x=alt.X("hours:Q", title="hours studied"),
        y=alt.Y("score:Q", title="exam score"),
        color=alt.Color("group:N", title="group"),
        tooltip=["hours:Q", "score:Q", "group:N"],
    )
    .properties(width=520, height=340, title="Encoding: x, y, colour, tooltip")
)
save_html(chart, LESSON, "encoding")

chart


## 2. The type suffix is not decoration

`:T` and `:N` on the same column give two different charts. The type suffix is not decoration: it tells Altair whether the values are ordered, and by how much.


In [ ]:
monthly = sales()

# :T tells Altair "this is time", so it gets a proper date axis.
correct = (
    alt.Chart(monthly).mark_line()
    .encode(x=alt.X("month:T", title="month"), y="sales:Q", color="region:N")
    .properties(width=520, height=260, title="month:T -- a real time axis")
)

# :N treats each date as an unrelated label: 24 categories, no ordering,
# equal spacing whatever the gaps.
wrong = (
    alt.Chart(monthly).mark_line()
    .encode(x=alt.X("month:N", title="month"), y="sales:Q", color="region:N")
    .properties(width=520, height=260, title="month:N -- 24 unrelated labels")
)
both_types = alt.vconcat(correct, wrong)
save_html(both_types, LESSON, "types")

both_types


## 3. Linked selection: the thing altair is genuinely best at

The thing Altair is genuinely best at. `selection_interval` plus `transform_filter` links two charts in four lines — brush the top one and the bottom one follows.


In [ ]:
brush = alt.selection_interval(encodings=["x"])

upper = (
    alt.Chart(monthly).mark_line()
    .encode(x="month:T", y="sales:Q", color="region:N")
    .properties(width=520, height=200, title="Drag across this chart...")
    .add_params(brush)
)

lower = (
    alt.Chart(monthly).mark_bar()
    .encode(x="sum(sales):Q", y=alt.Y("region:N", sort="-x"), color="region:N")
    .properties(width=520, height=160, title="...and this one follows")
    .transform_filter(brush)
)

linked = alt.vconcat(upper, lower)
save_html(linked, LESSON, "linked-selection")

linked


## Rules of thumb

```text
Altair makes you say what each column IS:
  :Q quantitative   :N nominal   :O ordinal   :T temporal
Get :T wrong and your dates become 24 unrelated categories.
Its real strength is linked views -- brush one chart, filter another.
In Streamlit:  st.altair_chart(chart, use_container_width=True)
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Change `group:N` to `group:O` in section 1. What changes in the legend, and what does that imply about the data?
2. In section 3, brush the top chart and watch the bars. Now add a third linked chart showing the mean score for the brushed range.
3. Encode `size` as well as `colour` in section 1. At what point does the chart have too many encodings?


In [ ]:
# your turn


---

**Previous:** [`interactive/plotly_basics`](plotly_basics.ipynb)  
**Next:** [`networks/networkx_basics`](../networks/networkx_basics.ipynb)
